# Transformer attention and causal masking

This notebook uses a tiny CPU example so every matrix is inspectable. GPU experiments will be added only when CUDA infrastructure exists.

In [ ]:
import math
import torch

from cuda_attention.attention import explicit_causal_attention

query = torch.tensor([[[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]]])
key = torch.tensor([[[[1.0, 0.0], [0.0, 1.0], [1.0, -1.0]]]])
value = torch.tensor([[[[10.0, 0.0], [0.0, 20.0], [30.0, 30.0]]]])

assert query.shape == key.shape == value.shape == (1, 1, 3, 2)
query, key, value

## Scores and scaling

`K.transpose(-2, -1)` changes `[S, D]` into `[D, S]`, so each query is compared with every key. Dividing by `sqrt(D)` controls dot-product magnitude.

In [ ]:
scores = query @ key.transpose(-2, -1)
scaled_scores = scores / math.sqrt(query.shape[-1])
assert scores.shape == (1, 1, 3, 3)
scores, scaled_scores

## Causal masking

Row `i` may use only columns `j <= i`. Future scores become negative infinity before softmax, producing exactly zero probability.

In [ ]:
allowed = torch.ones(3, 3, dtype=torch.bool).tril()
masked_scores = scaled_scores.masked_fill(~allowed, -torch.inf)
probabilities = torch.softmax(masked_scores, dim=-1)
output = probabilities @ value

assert torch.count_nonzero(probabilities.masked_select(~allowed)) == 0
torch.testing.assert_close(probabilities.sum(dim=-1), torch.ones(1, 1, 3))
allowed, masked_scores, probabilities, output

In [ ]:
reference = explicit_causal_attention(query, key, value)
torch.testing.assert_close(reference.probabilities, probabilities)
torch.testing.assert_close(reference.output, output)
reference

## Exercises

1. Predict which values can influence output row 0.
2. Change only the final value vector and explain which earlier outputs must remain unchanged.
3. Derive the flattened query position for rows 0 through 5 when `S = 3`.

TODO(student): Explain the shape of each intermediate tensor in your own words.

TODO(student): Explain why masking must occur before softmax.